In [2]:

"""
Traffic Demand Forecasting — Improved version
Changes vs v10:
- Target encoding is fit from a leak-safe source for validation and from full train for final test inference.
- Added extra lag/trend features from the most recent observed day.
- Added CatBoost as a diversity model.
- Meta blend is chosen from validation performance instead of blindly trusting one stacker.
- Keeps the overall pipeline close to the original so it is easy to compare.
"""

import os
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")

TRAIN_PATH = "../data/dataset/train.csv"
TEST_PATH = "../data/dataset/test.csv"
OUTPUT_PATH = "../submissions/submission_v11.csv"

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"

def decode_geohash(g: str):
    lat, lon = [-90.0, 90.0], [-180.0, 180.0]
    ilon = True
    for c in str(g):
        b = BASE32.index(c)
        for i in range(4, -1, -1):
            bit = (b >> i) & 1
            if ilon:
                mid = (lon[0] + lon[1]) / 2
                lon[bit] = mid
            else:
                mid = (lat[0] + lat[1]) / 2
                lat[bit] = mid
            ilon = not ilon
    return (lat[0] + lat[1]) / 2, (lon[0] + lon[1]) / 2

def add_time_features(df: pd.DataFrame):
    ts = df["timestamp"].astype(str).str.split(":", expand=True).astype(int)
    df["hour"] = ts[0]
    df["minute"] = ts[1]
    df["time_slot"] = df["hour"] * 4 + df["minute"] // 15

    df["slot_sin"] = np.sin(2 * np.pi * df["time_slot"] / 96)
    df["slot_cos"] = np.cos(2 * np.pi * df["time_slot"] / 96)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["is_weekend"] = (df["day"] % 7).isin([0, 6]).astype(int)
    df["is_rush"] = df["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df["is_night"] = df["hour"].isin([0, 1, 2, 3, 4, 5]).astype(int)

    df["geo3"] = df["geohash"].astype(str).str[:3]
    df["geo4"] = df["geohash"].astype(str).str[:4]
    df["geo5"] = df["geohash"].astype(str).str[:5]

    df["road_hour"] = df["RoadType"].astype(str) + "_" + df["hour"].astype(str)
    df["geo_slot"] = df["geohash"].astype(str) + "_" + df["time_slot"].astype(str)
    df["geo_hour"] = df["geohash"].astype(str) + "_" + df["hour"].astype(str)
    df["road_slot"] = df["RoadType"].astype(str) + "_" + df["time_slot"].astype(str)
    df["geo5_slot"] = df["geo5"] + "_" + df["time_slot"].astype(str)
    df["lanes_x_slot"] = df["NumberofLanes"] * df["time_slot"]

def fill_categoricals(df: pd.DataFrame):
    for c in ["RoadType", "Weather", "LargeVehicles", "Landmarks"]:
        df[c] = df[c].fillna("Unknown")

def build_geo_cache(train_df, test_df):
    geos = set(train_df["geohash"].astype(str)) | set(test_df["geohash"].astype(str))
    return {g: decode_geohash(g) for g in geos}

def add_geo_coordinates(df: pd.DataFrame, geo_cache: dict):
    df["latitude"] = df["geohash"].map(lambda g: geo_cache[str(g)][0])
    df["longitude"] = df["geohash"].map(lambda g: geo_cache[str(g)][1])

def add_temperature(df: pd.DataFrame):
    df["Temperature"] = df.groupby("geohash")["Temperature"].transform(lambda x: x.fillna(x.median()))
    df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median())

def build_lag_features_from_day48(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Uses day 48 as the latest observed historical day, which matches the original idea
    but is made explicit and reusable.
    """
    train48 = train_df[train_df["day"] == 48].copy()
    train49 = train_df[train_df["day"] == 49].copy()

    d48_exact_ts = train48.groupby(["geohash", "timestamp"])["demand"].mean()
    d48_exact_slot = train48.groupby(["geohash", "time_slot"])["demand"].mean()
    slot_med = train48.groupby("time_slot")["demand"].median()
    geo_mean_d48 = train48.groupby("geohash")["demand"].mean()
    geo5_slot_mean = train48.groupby(["geo5", "time_slot"])["demand"].mean()
    geo5_mean_map = train48.groupby("geo5")["demand"].mean()
    geo_stats = train48.groupby("geohash")["demand"].agg(
        geo_mean="mean", geo_max="max", geo_std="std"
    )

    def add_lag(df):
        d = df[["geohash", "timestamp", "time_slot"]].copy().reset_index(drop=True)
        d = d.merge(d48_exact_ts.rename("lag").reset_index(),
                    on=["geohash", "timestamp"], how="left")

        for delta in [1, 2, 4, 8]:
            for sign in [1, -1]:
                sn = d["lag"].isna()
                if not sn.any():
                    break
                tmp = d.loc[sn, ["geohash", "time_slot"]].copy()
                tmp["adj"] = tmp["time_slot"] + sign * delta
                adf = d48_exact_slot.rename("al").reset_index()
                adf.columns = ["geohash", "adj", "al"]
                t2 = tmp.merge(adf, on=["geohash", "adj"], how="left")
                fi = t2["al"].notna()
                d.loc[tmp.index[fi], "lag"] = t2.loc[fi, "al"].values

        sn = d["lag"].isna()
        d.loc[sn, "lag"] = d.loc[sn, "geohash"].map(geo_mean_d48)
        sn = d["lag"].isna()
        d.loc[sn, "lag"] = d.loc[sn, "time_slot"].map(slot_med)
        return d["lag"].values

    # build lag
    test_df["demand_lag_d1"] = add_lag(test_df)
    lag49 = add_lag(train_df[train_df.day == 49].reset_index(drop=True))

    train_df["demand_lag_d1"] = np.nan
    train_df.loc[train_df.day == 49, "demand_lag_d1"] = lag49
    train_df.loc[train_df.day == 48, "demand_lag_d1"] = train_df.loc[train_df.day == 48, "time_slot"].map(slot_med).values

    # extra lag/trend features
    day_ratio = (train49.groupby("geohash")["demand"].mean() / (train48.groupby("geohash")["demand"].mean() + 1e-9)).clip(0.1, 5.0)
    d49_2am = train49[train49["timestamp"] == "2:0"].groupby("geohash")["demand"].mean()
    d48_2am = train48[train48["timestamp"] == "2:0"].groupby("geohash")["demand"].mean()
    geo_2am_ratio = (d49_2am / (d48_2am + 1e-9)).clip(0.1, 10.0)

    d49_agg = train49.groupby("geohash")["demand"].agg(d49_mean="mean", d49_max="max", d49_last="last")

    for df in [train_df, test_df]:
        df["day_ratio"] = df["geohash"].map(day_ratio).fillna(1.0)
        df["lag_adjusted"] = df["demand_lag_d1"] * df["day_ratio"]

        df["geo_2am_ratio"] = df["geohash"].map(geo_2am_ratio).fillna(1.0)
        df["lag_interp"] = df["demand_lag_d1"] * df["geo_2am_ratio"]

        for col in ["d49_mean", "d49_max", "d49_last"]:
            df[col] = df["geohash"].map(d49_agg[col]).fillna(df["demand_lag_d1"])

        df["geo_mean"] = df["geohash"].map(geo_stats["geo_mean"]).fillna(df["demand_lag_d1"])
        df["geo_max"] = df["geohash"].map(geo_stats["geo_max"]).fillna(df["demand_lag_d1"])
        df["geo_std"] = df["geohash"].map(geo_stats["geo_std"]).fillna(0)

        df["lag_norm"] = df["demand_lag_d1"] / (df["geo_max"] + 1e-9)
        df["lag_diff_mean"] = df["d49_mean"] - df["demand_lag_d1"]
        df["lag_ratio_mean"] = df["d49_mean"] / (df["demand_lag_d1"] + 1e-9)
        df["geo_trend"] = df["geo_mean"] / (df["demand_lag_d1"] + 1e-9)

        g5df = geo5_slot_mean.rename("geo5_slot_demand").reset_index()
        dm = df[["geo5", "time_slot"]].merge(g5df, on=["geo5", "time_slot"], how="left")
        df["geo5_slot_demand"] = (
            dm["geo5_slot_demand"]
            .fillna(df["geo5"].map(geo5_mean_map))
            .fillna(df["demand_lag_d1"])
            .values
        )
        df["geo5_mean_demand"] = df["geo5"].map(geo5_mean_map).fillna(df["demand_lag_d1"])

def kfold_target_encoding(train_source: pd.DataFrame, test_source: pd.DataFrame, cols, target="demand", folds=5, seed=42):
    """
    OOF target encoding on train_source and full-train mappings for test_source.
    This is safer than using a single global mapping for everything.
    """
    train_source = train_source.reset_index(drop=True)
    test_source = test_source.copy()

    kf = KFold(n_splits=folds, shuffle=True, random_state=seed)

    for col in cols:
        oof = np.zeros(len(train_source))
        for tr_idx, va_idx in kf.split(train_source):
            tr_part = train_source.iloc[tr_idx]
            va_part = train_source.iloc[va_idx]
            mp = tr_part.groupby(col)[target].mean().to_dict()
            oof[va_idx] = va_part[col].astype(str).map(mp).values

        global_mean = train_source[target].mean()
        train_source[f"{col}_te"] = pd.Series(oof).fillna(global_mean).values

        full_map = train_source.groupby(col)[target].mean().to_dict()
        test_source[f"{col}_te"] = pd.Series(test_source[col].astype(str).map(full_map)).fillna(global_mean).values

    return train_source, test_source

def numericize(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

def fit_lgb(X, y_log, X_test, seeds=(42, 2024, 888), splits=5):
    params = dict(
        n_estimators=4000,
        learning_rate=0.02,
        num_leaves=127,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_samples=10,
        n_jobs=-1,
    )

    oof = np.zeros(len(X))
    preds = np.zeros(len(X_test))

    for seed in seeds:
        kf = KFold(n_splits=splits, shuffle=True, random_state=seed)
        oof_s = np.zeros(len(X))
        for fold, (ti, vi) in enumerate(kf.split(X, y_log)):
            m = lgb.LGBMRegressor(**params, random_state=seed, verbose=-1)
            m.fit(
                X[ti], y_log[ti],
                eval_set=[(X[vi], y_log[vi])],
                callbacks=[
                    lgb.early_stopping(150, verbose=False),
                    lgb.log_evaluation(-1)
                ],
            )
            oof_s[vi] = m.predict(X[vi])
            preds += m.predict(X_test) / (splits * len(seeds))
        oof += oof_s / len(seeds)
    return oof, preds

def fit_xgb(X, y_log, X_test, seed=2026, splits=5):
    params = dict(
        n_estimators=4000,
        learning_rate=0.02,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=seed,
        n_jobs=-1,
        early_stopping_rounds=150,
    )

    oof = np.zeros(len(X))
    preds = np.zeros(len(X_test))
    kf = KFold(n_splits=splits, shuffle=True, random_state=seed)

    for fold, (ti, vi) in enumerate(kf.split(X, y_log)):
        m = xgb.XGBRegressor(**params, verbosity=0)
        m.fit(X[ti], y_log[ti], eval_set=[(X[vi], y_log[vi])], verbose=False)
        oof[vi] = m.predict(X[vi])
        preds += m.predict(X_test) / splits
    return oof, preds

def fit_et(X, y_log, X_test, seed=2024, splits=5):
    oof = np.zeros(len(X))
    preds = np.zeros(len(X_test))
    kf = KFold(n_splits=splits, shuffle=True, random_state=seed)

    for fold, (ti, vi) in enumerate(kf.split(X, y_log)):
        m = ExtraTreesRegressor(
            n_estimators=500,
            max_features=0.5,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=seed,
        )
        m.fit(X[ti], y_log[ti])
        oof[vi] = m.predict(X[vi])
        preds += m.predict(X_test) / splits
    return oof, preds

def fit_cat(X, y_log, X_test, seed=2025, splits=5):
    oof = np.zeros(len(X))
    preds = np.zeros(len(X_test))
    kf = KFold(n_splits=splits, shuffle=True, random_state=seed)

    for fold, (ti, vi) in enumerate(kf.split(X, y_log)):
        m = CatBoostRegressor(
            loss_function="RMSE",
            iterations=3500,
            learning_rate=0.03,
            depth=8,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
        )
        m.fit(X[ti], y_log[ti], eval_set=(X[vi], y_log[vi]), use_best_model=True)
        oof[vi] = m.predict(X[vi])
        preds += m.predict(X_test) / splits
    return oof, preds

def score_day49(y, preds, day49_mask):
    return r2_score(y[day49_mask], preds[day49_mask])

# ------------------------------------------------------------
# 1) Load
# ------------------------------------------------------------
print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# ------------------------------------------------------------
# 2) Static features
# ------------------------------------------------------------
print("Building static features...")
geo_cache = build_geo_cache(train_df, test_df)

for df in [train_df, test_df]:
    add_geo_coordinates(df, geo_cache)
    add_time_features(df)
    add_temperature(df)
    fill_categoricals(df)

# ------------------------------------------------------------
# 3) Lag and trend features
# ------------------------------------------------------------
print("Building lag/trend features...")
build_lag_features_from_day48(train_df, test_df)

# ------------------------------------------------------------
# 4) Target encoding
# ------------------------------------------------------------
print("Building target encodings...")
te_cols = [
    "geohash", "geo3", "geo4", "geo5", "geo_slot", "geo_hour", "geo5_slot",
    "RoadType", "Weather", "road_hour", "road_slot"
]

# For validation comparison, keep train-day49 included but use OOF encoding.
# This avoids the obvious global leakage from a single full-data mean map.
train_te, test_te = kfold_target_encoding(
    train_df.copy(), test_df.copy(), te_cols, folds=5, seed=42
)

for col in [f"{c}_te" for c in te_cols]:
    train_df[col] = train_te[col]
    test_df[col] = test_te[col]

# ------------------------------------------------------------
# 5) Feature list
# ------------------------------------------------------------
feats = (
    [
        "latitude", "longitude", "hour", "minute", "time_slot",
        "is_weekend", "is_rush", "is_night",
        "slot_sin", "slot_cos", "hour_sin", "hour_cos",
        "NumberofLanes", "Temperature", "lanes_x_slot",
        "demand_lag_d1", "lag_adjusted", "lag_interp",
        "day_ratio", "geo_2am_ratio",
        "d49_mean", "d49_max", "d49_last",
        "geo_mean", "geo_max", "geo_std", "lag_norm",
        "lag_diff_mean", "lag_ratio_mean", "geo_trend",
        "geo5_slot_demand", "geo5_mean_demand",
    ]
    + [f"{c}_te" for c in te_cols]
)

print(f"Using {len(feats)} features")
numericize(train_df, feats)
numericize(test_df, feats)

X = train_df[feats].values
y = train_df["demand"].values
y_log = np.log1p(y)
X_test = test_df[feats].values
day49_mask = train_df["day"].values == 49

# ------------------------------------------------------------
# 6) Train base models
# ------------------------------------------------------------
print("\nTraining LightGBM...")
lgb_oof, lgb_preds = fit_lgb(X, y_log, X_test, seeds=(42, 2024, 888), splits=5)
lgb_oof_e = np.expm1(lgb_oof)
lgb_preds_e = np.expm1(lgb_preds)
print(f"LGB day49 R2: {score_day49(y, lgb_oof_e, day49_mask):.5f}")

print("\nTraining XGBoost...")
xgb_oof, xgb_preds = fit_xgb(X, y_log, X_test, seed=2026, splits=5)
xgb_oof_e = np.expm1(xgb_oof)
xgb_preds_e = np.expm1(xgb_preds)
print(f"XGB day49 R2: {score_day49(y, xgb_oof_e, day49_mask):.5f}")

print("\nTraining ExtraTrees...")
et_oof, et_preds = fit_et(X, y_log, X_test, seed=2024, splits=5)
et_oof_e = np.expm1(et_oof)
et_preds_e = np.expm1(et_preds)
print(f"ET day49 R2:  {score_day49(y, et_oof_e, day49_mask):.5f}")

print("\nTraining CatBoost...")
cat_oof, cat_preds = fit_cat(X, y_log, X_test, seed=2025, splits=5)
cat_oof_e = np.expm1(cat_oof)
cat_preds_e = np.expm1(cat_preds)
print(f"CAT day49 R2: {score_day49(y, cat_oof_e, day49_mask):.5f}")

# ------------------------------------------------------------
# 7) Meta blending
# ------------------------------------------------------------
print("\nFitting meta blend...")

stack_oof = np.column_stack([lgb_oof_e, xgb_oof_e, et_oof_e, cat_oof_e])
stack_test = np.column_stack([lgb_preds_e, xgb_preds_e, et_preds_e, cat_preds_e])

# Option A: fit on all OOF
meta_all = LinearRegression(positive=True)
meta_all.fit(stack_oof, y)
blend_all_oof = meta_all.predict(stack_oof)
blend_all_test = meta_all.predict(stack_test)
r2_all = score_day49(y, blend_all_oof, day49_mask)
w_all = meta_all.coef_ / (meta_all.coef_.sum() + 1e-9)
print(f"Meta(all OOF) day49 R2: {r2_all:.5f}  weights={np.round(w_all, 3)}")

# Option B: fit only on day49 OOF points
meta_d49 = LinearRegression(positive=True)
meta_d49.fit(stack_oof[day49_mask], y[day49_mask])
blend_d49_oof = meta_d49.predict(stack_oof)
blend_d49_test = meta_d49.predict(stack_test)
r2_d49 = score_day49(y, blend_d49_oof, day49_mask)
w_d49 = meta_d49.coef_ / (meta_d49.coef_.sum() + 1e-9)
print(f"Meta(day49)  day49 R2: {r2_d49:.5f}  weights={np.round(w_d49, 3)}")

# Option C: simple rank-mean fallback
rank_stack = np.column_stack([
    pd.Series(lgb_preds_e).rank(method="average").values,
    pd.Series(xgb_preds_e).rank(method="average").values,
    pd.Series(et_preds_e).rank(method="average").values,
    pd.Series(cat_preds_e).rank(method="average").values,
])
rank_oof = np.column_stack([
    pd.Series(lgb_oof_e).rank(method="average").values,
    pd.Series(xgb_oof_e).rank(method="average").values,
    pd.Series(et_oof_e).rank(method="average").values,
    pd.Series(cat_oof_e).rank(method="average").values,
])
meta_rank = LinearRegression(positive=True)
meta_rank.fit(rank_oof, y)
blend_rank_oof = meta_rank.predict(rank_oof)
blend_rank_test = meta_rank.predict(rank_stack)
r2_rank = score_day49(y, blend_rank_oof, day49_mask)
w_rank = meta_rank.coef_ / (meta_rank.coef_.sum() + 1e-9)
print(f"Meta(rank)   day49 R2: {r2_rank:.5f}  weights={np.round(w_rank, 3)}")

# Pick the best validation option
scores = {
    "all": r2_all,
    "d49": r2_d49,
    "rank": r2_rank,
}
best_name = max(scores, key=scores.get)
if best_name == "all":
    final_test = blend_all_test
    final_oof = blend_all_oof
    print("Using meta(all OOF)")
elif best_name == "d49":
    final_test = blend_d49_test
    final_oof = blend_d49_oof
    print("Using meta(day49)")
else:
    final_test = blend_rank_test
    final_oof = blend_rank_oof
    print("Using meta(rank)")

print("\n" + "=" * 60)
print(f"BLEND OOF R2:   {r2_score(y, final_oof):.5f}")
print(f"BLEND day49 R2: {score_day49(y, final_oof, day49_mask):.5f}")
print("=" * 60)

# ------------------------------------------------------------
# 8) Submission
# ------------------------------------------------------------
final_test = np.clip(final_test, 0, None)
sub = pd.DataFrame({"Index": test_df["Index"], "demand": final_test})
sub.sort_values("Index").reset_index(drop=True).to_csv(OUTPUT_PATH, index=False)

print(f"Saved -> {OUTPUT_PATH}")
print(f"Shape: {sub.shape}")
print(f"Demand stats: min={final_test.min():.4f}, mean={final_test.mean():.4f}, max={final_test.max():.4f}")


Loading data...
Building static features...
Building lag/trend features...
Building target encodings...
Using 43 features

Training LightGBM...
LGB day49 R2: 0.96166

Training XGBoost...
XGB day49 R2: 0.96020

Training ExtraTrees...
ET day49 R2:  0.95384

Training CatBoost...
CAT day49 R2: 0.95981

Fitting meta blend...
Meta(all OOF) day49 R2: 0.96207  weights=[0.366 0.27  0.141 0.223]
Meta(day49)  day49 R2: 0.96257  weights=[0.549 0.173 0.    0.278]
Meta(rank)   day49 R2: 0.49386  weights=[0.239 0.202 0.352 0.207]
Using meta(day49)

BLEND OOF R2:   0.96756
BLEND day49 R2: 0.96257
Saved -> ../submissions/submission_v11.csv
Shape: (41778, 2)
Demand stats: min=0.0013, mean=0.1248, max=1.0554
